# Experiment 2 (control) — Conflict test with the matched-baseline probe

Same as `02_conflict.ipynb`, but reads the probe directions from the **matched-baseline control** Exp 1 bundle (`exp1_authority_control_matched`) instead of the raw `exp1_authority` bundle.

Motivation: the raw Exp 1 S/U contrast has a huge surface feature (presence/absence of the system block), so its probe direction behaves mostly as a structural-position detector in Exp 2. The control was trained with a matched baseline system block on both sides, so only the placement of the instruction-of-interest varies.

This run probes **every layer**, not just the best one. For each row we store the projection onto each layer's probe direction and count how many layers "activate" (project above that layer's own median). The hypothesis: more layers activated → stronger authority signal propagating through the residual stream → more likely the model follows the system.

**Prerequisite:** `run_experiment_1_control` has already been run for the chosen model and its bundle lives on Drive at `…/<slug>/exp1_authority_control_matched/` (adjust `EXP1_CONTROL_DIRNAME` in the config cell if yours is named differently).

In [ ]:
# Clone repo (for data/ + tests/) and install it in editable mode
import os, sys, importlib, site
REPO_DIR = "/content/Mech_spoof"

if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/ChuloIva/Mech_spoof.git {REPO_DIR}

!pip install -q -e {REPO_DIR}

# Make sure mech_spoof resolves data/ to the cloned repo
os.environ["MECH_SPOOF_ROOT"] = REPO_DIR

# Ensure the current kernel picks up the freshly installed package
# (Colab caches sys.path before pip runs; editable installs need a nudge)
src_path = f"{REPO_DIR}/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()
site.main()

import mech_spoof
print("mech_spoof loaded from:", mech_spoof.__file__)

In [ ]:
# Auth + Drive
import os, getpass
from google.colab import drive, userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
try:
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
except Exception:
    pass
if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ').strip()
drive.mount('/content/drive')

In [ ]:
!pip install --upgrade transformers

In [ ]:
# Fallback key prompt (run if the Drive userdata fetch above didn't populate OPENROUTER_API_KEY)
import os, getpass
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter key: ').strip()
print('len:', len(os.environ['OPENROUTER_API_KEY']))  # non-zero confirms set

In [ ]:
# Config (EDIT ME)
from pathlib import Path
from mech_spoof.configs import MODEL_CONFIGS

MODEL_KEY = 'qwen'            # qwen | llama3 | mistral | gemma | phi3
DRIVE_ROOT = Path('/content/drive/MyDrive/mech_spoof_results')

# Where the matched-baseline Exp 1 bundle lives on Drive (written by run_experiment_1_control).
EXP1_CONTROL_DIRNAME = 'exp1_authority_control_matched'
# Where this run's Exp 2 outputs go (kept separate from the regular exp2 bundle).
EXP2_CONTROL_DIRNAME = 'exp2_conflict_control'

slug = MODEL_CONFIGS[MODEL_KEY].slug
EXP1_DIR = DRIVE_ROOT / slug / EXP1_CONTROL_DIRNAME
OUT_DIR  = DRIVE_ROOT / slug / EXP2_CONTROL_DIRNAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('EXP1 control dir :', EXP1_DIR, '(exists:', EXP1_DIR.exists(), ')')
print('OUT_DIR          :', OUT_DIR)

assert EXP1_DIR.exists(), (
    f'Control Exp 1 bundle not found at {EXP1_DIR}. Run `run_experiment_1_control` for '
    f'this model first, or adjust EXP1_CONTROL_DIRNAME above.'
)

In [ ]:
# Peek at the control probe we're about to use
from mech_spoof.io import load_json
ctrl = load_json(EXP1_DIR / 'result.json')
print('variant     :', ctrl.get('variant'))
print('baseline_sys:', ctrl.get('baseline_system'))
print('best_layer  :', ctrl.get('best_layer'))
print('best_acc    :', round(ctrl.get('best_accuracy', float('nan')), 4))
accs = ctrl.get('accuracies') or {}
print('per-layer accuracies (first 10):', {int(k): round(float(v), 3) for k, v in list(accs.items())[:10]})

In [ ]:
# Run Exp 2 using the control probe's directions (all layers)
from mech_spoof.experiments.exp2_conflict import run_experiment_2
result = run_experiment_2(MODEL_KEY, OUT_DIR, exp1_dir=EXP1_DIR)
print('done:', OUT_DIR)

In [ ]:
# Headline numbers (best-layer correlations are back-compat with the original exp2)
import json
print('=== summary (includes mean n_layers_activated per condition) ===')
print(json.dumps(result.summary, indent=2))
print('\n=== correlation — best-layer (overall + per condition) ===')
for key in ('overall', 'REAL', 'NONE', 'FAKE'):
    v = result.correlation.get(key)
    if v:
        print(f'  {key:<7}  r={v["r"]:+.4f}  p={v["p"]:.4f}  n={v["n"]}')

print('\n=== correlation — n_layers_activated (coarse count of layers above own median) ===')
for field in ('n_activated_above_mean', 'n_activated_strong'):
    sub = result.correlation.get(field) or {}
    if not sub:
        continue
    print(f'  {field}:')
    for key in ('overall', 'REAL', 'NONE', 'FAKE'):
        v = sub.get(key)
        if v:
            print(f'    {key:<7}  r={v["r"]:+.4f}  p={v["p"]:.4f}  n={v["n"]}')

In [ ]:
# Per-layer correlation curve — which depth (if any) best predicts compliance?
import matplotlib.pyplot as plt

by_layer = (result.correlation.get('by_layer') or {})
if not by_layer:
    print('No per-layer correlations available (probe directions missing or too few judged rows).')
else:
    layers = sorted(by_layer.keys())

    def _r(layer, cond):
        v = by_layer.get(layer, {}).get(cond)
        return v['r'] if v else float('nan')

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(layers, [_r(l, 'overall') for l in layers], marker='o', label='overall')
    ax.plot(layers, [_r(l, 'REAL')    for l in layers], marker='s', alpha=0.7, label='REAL')
    ax.plot(layers, [_r(l, 'NONE')    for l in layers], marker='^', alpha=0.7, label='NONE')
    ax.plot(layers, [_r(l, 'FAKE')    for l in layers], marker='v', alpha=0.7, label='FAKE')
    ax.axhline(0, color='gray', ls='--', lw=0.8)
    ax.set_xlabel('layer')
    ax.set_ylabel('Pearson r (probe score vs system_followed)')
    ax.set_title(f'{MODEL_KEY} — per-layer probe/compliance correlation (matched-baseline probe)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'per_layer_correlation.png', dpi=120)
    plt.show()

    # Top 5 layers by |overall r|
    ranked = sorted(
        (l for l in layers if by_layer[l].get('overall')),
        key=lambda l: -abs(by_layer[l]['overall']['r']),
    )
    print('Top layers by |r| (overall):')
    for l in ranked[:5]:
        v = by_layer[l]['overall']
        print(f'  layer {l:>3}  r={v["r"]:+.4f}  p={v["p"]:.4f}  n={v["n"]}')